# 2D surface roughness via Trachytopes

Sets up spatially-uniform **Manning** roughness on the 2D surface of the model
built by `template.ipynb`, using D-Flow FM's *trachytopes* module.

hydrolib-core has no `.ttd`/`.arl` writer — on the `FMModel` you only set paths:

```python
fm.trachytopes.trtrou = "Y"
fm.trachytopes.trtdef = "roughness.ttd"   # class definitions
fm.trachytopes.trtl   = "roughness.arl"   # class -> 2D net-link mapping
fm.trachytopes.dttrt  = fm.time.dtuser    # must be a multiple of DtUser
```

The two plain-text kernel files (D-Flow FM User Manual App. C.8) are authored
below and written into the exported model's `dflowfm/` folder:

- **`roughness.ttd`** — the class *definitions*: `1 53 0.10` = trachytope class 1,
  formula 53 (Manning), n = 0.10.
- **`roughness.arl`** — the *distribution*: one line `xu yu zu TrachytopeNr Fraction`
  per 2D net link, assigning class 1 over the whole link (fraction 1.0).

> Run `template.ipynb` first so `delft3d_model` and its `network.nc` exist.


In [1]:
from pathlib import Path
import shutil

import xarray as xr

# The template model must have been exported first (template.ipynb).
data_path = Path("../tests/data/template/datasets").resolve()
output_path = data_path / "delft3d_model"
dfm_dir = output_path / "dflowfm"
ttd_src = data_path / "trachytopes" / "roughness.ttd"   # dataset roughness data

assert dfm_dir.exists(), "Run template.ipynb first to export delft3d_model"
assert ttd_src.exists(), f"Missing roughness definitions: {ttd_src}"
print("model dir      :", dfm_dir)
print("roughness class:", ttd_src.read_text().strip().splitlines()[-1])


model dir      : G:\workspace\github\HYDROLIB\hydrolib\tests\data\template\datasets\delft3d_model\dflowfm
roughness class: 1 53 0.10


In [2]:
def write_trachytope_files(dfm_dir, ttd_src):
    """Write the two trachytope files into a model's dflowfm dir.

    - roughness.ttd : the Manning class definition (copied from the dataset).
    - roughness.arl : maps that class onto every 2D net link. Net-link midpoints
      are read from network.nc (mesh2d_edge_x/y) so the .arl coordinates match
      exactly what the kernel reads.
    """
    dfm_dir = Path(dfm_dir)
    shutil.copyfile(ttd_src, dfm_dir / "roughness.ttd")
    with xr.open_dataset(dfm_dir / "network.nc") as ds:
        xu = ds["mesh2d_edge_x"].values
        yu = ds["mesh2d_edge_y"].values
    out = ["# Area Roughness on Links - Manning class 1 on every 2D net link",
           "# xu yu zu TrachytopeNr Fraction"]
    out += [f"{x:.6f} {y:.6f} 0 1 1.0" for x, y in zip(xu, yu)]
    (dfm_dir / "roughness.arl").write_text("\n".join(out) + "\n")
    return len(xu)


n = write_trachytope_files(dfm_dir, ttd_src)
print(f"wrote roughness.ttd + roughness.arl ({n} 2D links, Manning n=0.10)")


wrote roughness.ttd + roughness.arl (9655 2D links, Manning n=0.10)


In [3]:
def enable_trachytopes(dfm_dir):
    """Switch the [Trachytopes] section on in the exported DFM.mdu (equivalent to
    the fm.trachytopes.* assignments). dtTrt is set to DtUser, since the kernel
    requires it to be a multiple of DtUser."""
    mdu = Path(dfm_dir) / "DFM.mdu"
    lines = mdu.read_text().splitlines()

    dtuser = 300.0
    for ln in lines:
        if ln.split("=")[0].strip().lower() == "dtuser":
            dtuser = float(ln.split("=")[1].split("#")[0])
            break

    repl = {"trtrou": ("trtRou", "Y"),
            "trtdef": ("trtDef", "roughness.ttd"),
            "trtl":   ("trtL",   "roughness.arl"),
            "dttrt":  ("dtTrt",  f"{dtuser:.1f}")}
    for i, ln in enumerate(lines):
        key = ln.split("=")[0].strip().lower()
        if key in repl:
            name, val = repl[key]
            cmt = ln.split("#", 1)[1].strip() if "#" in ln else ""
            lines[i] = f"{name:<6} = {val:<15}" + (f"# {cmt}" if cmt else "")
    mdu.write_text("\n".join(lines) + "\n")


enable_trachytopes(dfm_dir)
print("Trachytopes enabled in DFM.mdu ([Trachytopes] trtRou=Y).")


Trachytopes enabled in DFM.mdu ([Trachytopes] trtRou=Y).
